# 1. Inisialisasi

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

warnings.filterwarnings('ignore')
print("Library import success!")

# 2. Akuisis Data

In [ ]:
data_path = "../dataset/online_shoppers_intention.csv"
if not os.path.exists(data_path):
    data_path = "https://raw.githubusercontent.com/okapucino/online-shoppers/main/dataset/online_shoppers_intention.csv"

df = pd.read_csv(data_path)
print(f"Dimensi Dataset Mentah: {df.shape[0]} Baris x {df.shape[1]} Kolom")
df.head()

# 3. Cleaning Data

In [ ]:
print("Total Missing Value:", df.isnull().sum().sum())

jml_dup = df.duplicated().sum()
print(f"Jumlah baris duplikat terdeteksi: {jml_dup} baris")
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dimensi dataset setelah de-duplikasi: {df.shape[0]} baris")

df['Revenue_Num'] = df['Revenue'].astype(int)

num_cols = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
cat_cols = ['Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']

print(f"Fitur Numerik ({len(num_cols)} fitur):", num_cols)
print(f"Fitur Categorical ({len(cat_cols)} fitur):", cat_cols)

# 4. Analisis Ekplorative & Class Imbalance

In [ ]:
plt.figure(figsize=(6.5, 4.5))
counts = df['Revenue'].value_counts()
plt.bar(['False', 'True'], counts.values, color=['#0390fc', '#ff7700'], width=0.45, edgecolor='black')
for idx, val in enumerate(counts.values):
    pct = (val / len(df)) * 100
    plt.text(idx, val + 150, f"{val:,} ({pct:.1f}%)", ha='center', fontweight='bold')
plt.title('Distribusi Kelas Target: Keputusan Pembelian Pengunjung', fontsize='12', fontweight='bold')
plt.ylabel('Jumlah Observasi Sesi')
plt.ylim(0, 12500)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.boxplot(data=df, x='Revenue', y='PageValues', hue='Revenue', palette=['#0390fc', '#ff7700'], legend=False, showfliers=False)
plt.title('Distribusi PageValues Berdasarkan Status Pembelian', fontsize=12, fontweight='bold')
plt.xticks([0, 1], ['Tidak Beli', 'Membeli'])
plt.ylabel('Nilai Halaman Rata-rata')
plt.tight_layout()
plt.show()

# 5 Encoding & Pembagian Data

In [ ]:
df_cat = pd.get_dummies(df[cat_cols].astype(str), drop_first=True)
X = pd.concat([df[num_cols], df_cat], axis=1)
y = df['Revenue_Num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(f"Data Latih: {X_train.shape[0]} baris | Data Uji: {X_test.shape[0]} baris | Fitur Total: {X.shape[1]} kolom")

# 6. Modeling Baseline vs Ensembale

In [ ]:
model_lr = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
model_lr.fit(X_train_scaled, y_train)
y_pred_lr = model_lr.predict(X_test_scaled)
y_prob_lr = model_lr.predict_proba(X_test_scaled)[:, 1]

model_rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)
y_prob_rf = model_rf.predict_proba(X_test)[:, 1]
print("Kedua model telah selesai dilatih!")

# 7. Evaluasi Perbandingan & Visualisasi Kinerja

In [ ]:
eval_df = pd.DataFrame({
    'Model': ['Baseline: Logistic Regression', 'Ensemble: Random Forest'],
    'Akurasi': [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_rf)],
    'Presisi': [precision_score(y_test, y_pred_lr), precision_score(y_test, y_pred_rf)],
    'Recall': [recall_score(y_test, y_pred_lr), recall_score(y_test, y_pred_rf)],
    'F1-Score': [f1_score(y_test, y_pred_lr), f1_score(y_test, y_pred_rf)],
    'ROC-AUC': [roc_auc_score(y_test, y_prob_lr), roc_auc_score(y_test, y_prob_rf)]
})
eval_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))

cm_lr = confusion_matrix(y_test, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Prediksi Tidak Beli', 'Prediksi Membeli'], yticklabels=['Aktual Tidak Beli', 'Aktual Membeli'])
axes[0].set_title(f"Baseline: Logistic Regression\nRecall: {recall_score(y_test, y_pred_lr):.3f} | Akurasi: {accuracy_score(y_test, y_pred_lr):.3f}", fontweight='bold')
axes[0].set_ylabel('Kelas Aktual')

cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=axes[1], cbar=False,
            xticklabels=['Prediksi Tidak Beli', 'Prediksi Membeli'], yticklabels=['Aktual Tidak Beli', 'Aktual Membeli'])
axes[1].set_title(f'Ensembale: Random Forest\nRecall: {recall_score(y_test, y_pred_rf):.3f} | Akurasi {accuracy_score(y_test, y_pred_rf):.3f}', fontweight='bold')
axes[1].set_ylabel('Kelas Aktual')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6.5, 5.5))
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})', color='#0390fc', lw=2)
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {roc_auc_score(y_test, y_prob_rf):.3f})', color="#ff7700", lw=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Guess (AUC = 0.500)')
plt.title('Kurva ROC: Komparasi Model Prediksi Pembelian', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8.5, 5))
imp = pd.Series(model_rf.feature_importances_, index=X.columns).sort_values(ascending=True).tail(10)
imp.plot(kind='barh', color='#04ff00', edgecolor='black')
plt.title('Top 10 Fitur Paling Berpengaruh Terhadap Pembelian (Random Forest)', fontsize=12, fontweight='bold')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()